In [8]:
hpi = pd.read_pickle("../data/processed/hpi_by_group_quarterly.pkl")

In [1]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd
import scipy.sparse as sp

pairs = pd.read_pickle("../data/processed/repeat_sales_pairs.pkl")

def qidx(d):                      # 0 = 2003Q1, 1 = 2003Q2, ...
    return ((d.dt.year - 2003) * 4 + (d.dt.quarter - 1)).to_numpy()

pairs['q0'], pairs['q1'] = qidx(pairs['date0']), qidx(pairs['date1'])
pairs = pairs[pairs['q1'] > pairs['q0']].reset_index(drop=True)   # same-quarter pairs say nothing about change
T = int(pairs['q1'].max()) + 1
print("pairs:", len(pairs), "| quarters:", T)

def design(p):
    n = len(p)
    rows = np.r_[np.arange(n), np.arange(n)]
    cols = np.r_[p['q0'].to_numpy(), p['q1'].to_numpy()]
    vals = np.r_[-np.ones(n), np.ones(n)]
    return sp.csr_matrix((vals, (rows, cols)), shape=(n, T))[:, 1:]   # drop first quarter: beta_0 = 0

def fit(p, w=None):
    D, y = design(p), p['log_ratio'].to_numpy()
    W = sp.diags(np.ones(len(p)) if w is None else w)
    A = (D.T @ W @ D).toarray()
    b = D.T @ (W @ y)
    beta = np.r_[0.0, np.linalg.lstsq(A, b, rcond=None)[0]]
    resid = y - D @ beta[1:]
    return beta, resid

# quick check: pairs per quarter touching each period (thin periods = noisy index)
touch = np.bincount(np.r_[pairs['q0'], pairs['q1']], minlength=T)
print("pairs touching the 8 thinnest quarters:", np.sort(touch)[:8])

pairs: 364334 | quarters: 92
pairs touching the 8 thinnest quarters: [3895 4404 4444 4532 4776 4886 4902 5022]


In [2]:
labels = [f"{2003 + i // 4}Q{i % 4 + 1}" for i in range(T)]
yrs = (pairs['days'] / 365.25).to_numpy()

# 1. BMN: plain OLS
b_ols, r1 = fit(pairs)

# 2. Case-Shiller weighting: model squared residuals as a + b*interval, weight by inverse fitted variance
X = np.c_[np.ones(len(pairs)), yrs]
coef = np.linalg.lstsq(X, r1**2, rcond=None)[0]
var = np.clip(X @ coef, 1e-4, None)
print("variance model: a =", coef[0].round(4), " b per year =", coef[1].round(4))
b_wls, r2 = fit(pairs, w=1 / var)

# 3. Robust trimming: drop pairs with |resid| > 3 robust SD, refit
sd = 1.4826 * np.median(np.abs(r2 - np.median(r2)))
keep = np.abs(r2) <= 3 * sd
print(f"robust SD {sd:.3f} | trimmed {np.sum(~keep):,} pairs ({np.mean(~keep):.1%})")
p2 = pairs[keep].reset_index(drop=True)
yrs2 = (p2['days'] / 365.25).to_numpy()
X2 = np.c_[np.ones(len(p2)), yrs2]
c2 = np.linalg.lstsq(X2, (fit(p2)[1])**2, rcond=None)[0]
b_trim, _ = fit(p2, w=1 / np.clip(X2 @ c2, 1e-4, None))

idx = pd.DataFrame({'ols': b_ols, 'wls': b_wls, 'wls_trim': b_trim}, index=labels)
idx = 100 * np.exp(idx)
idx = idx / idx.loc['2004Q1'] * 100          # rebase: 2003 is thin, so 2004Q1 = 100
idx['pairs_touching'] = touch

print(idx.loc[[l for l in labels if l.endswith('Q4')]].round(1).to_string())

variance model: a = 0.2494  b per year = -0.0082
robust SD 0.288 | trimmed 20,577 pairs (5.6%)
          ols    wls  wls_trim  pairs_touching
2003Q4   93.1   94.0      95.5            7348
2004Q4  112.9  113.5     114.8           11460
2005Q4  139.1  139.8     140.2           11751
2006Q4  151.8  151.4     151.1           11882
2007Q4  149.1  148.7     149.3            9351
2008Q4  137.3  138.0     139.2            7293
2009Q4  124.8  125.5     127.0            8617
2010Q4  117.5  118.0     118.8            5022
2011Q4  122.3  123.1     125.4            6076
2012Q4  130.4  131.0     134.4            7626
2013Q4  142.2  142.6     145.6            9401
2014Q4  153.0  153.1     156.5            8824
2015Q4  165.4  165.2     167.1            9130
2016Q4  174.1  173.7     175.8            8253
2017Q4  184.2  183.4     185.2            8423
2018Q4  193.1  191.9     192.9            7739
2019Q4  196.1  194.6     195.6            6943
2020Q4  208.8  206.4     198.7            6563
2021Q4  216.

In [3]:
idx.to_pickle("../data/processed/hpi_citywide_quarterly.pkl")

In [4]:
# A. Robust noise level by holding period (MAD-based SD, immune to the heavy tail)
b_t, r_t = fit(pairs, w=1 / var)
pairs['resid'] = r_t
pairs['gap'] = pd.cut(pairs['days'] / 365.25, [0, 0.5, 1, 2, 5, 10, 25])
rs = lambda s: 1.4826 * np.median(np.abs(s - np.median(s)))
print(pairs.groupby('gap', observed=True)['resid'].agg(n='size', robust_sd=rs).round(3).to_string())

# B. Which pairs does trimming remove? (by the year of the later sale)
pairs['trim'] = np.abs(pairs['resid']) > 3 * sd
t = pairs.groupby(pairs['date1'].dt.year).agg(trim_share=('trim', 'mean'), n=('trim', 'size'))
t['mean_resid_of_trimmed'] = pairs[pairs['trim']].groupby(pairs['date1'].dt.year)['resid'].mean()
t['share_positive'] = pairs[pairs['trim']].groupby(pairs['date1'].dt.year)['resid'].apply(lambda s: (s > 0).mean())
print(t.round(3).to_string())
print(pairs[pairs['trim'] & (pairs['date1'].dt.year == 2020)]
      .sample(10, random_state=1)[['SEGMENT', 'BOROUGH', 'date0', 'price0', 'date1', 'price1', 'resid']].to_string())

                   n  robust_sd
gap                            
(0.0, 0.5]     16049      0.335
(0.5, 1.0]     27666      0.400
(1.0, 2.0]     38350      0.314
(2.0, 5.0]     91804      0.225
(5.0, 10.0]   111068      0.260
(10.0, 25.0]   79397      0.301
       trim_share      n  mean_resid_of_trimmed  share_positive
date1                                                          
2003        0.037   1312                  0.554           0.776
2004        0.045   4923                  0.131           0.677
2005        0.032   9601                  0.344           0.693
2006        0.030  12999                  0.249           0.641
2007        0.030  13201                  0.433           0.701
2008        0.031  11800                 -0.013           0.528
2009        0.059  11406                 -0.085           0.453
2010        0.069  11208                 -0.038           0.473
2011        0.070  12100                 -0.031           0.488
2012        0.071  14066                

In [5]:
def pipeline(p, min_touch=30):
    p = p.reset_index(drop=True)
    b1, r1 = fit(p)
    s1_ = 1.4826 * np.median(np.abs(r1 - np.median(r1)))
    ok = np.abs(r1) <= 3 * s1_
    X = np.c_[np.ones(len(p)), (p['days'] / 365.25).to_numpy()]
    c = np.linalg.lstsq(X[ok], r1[ok]**2, rcond=None)[0]
    b2, r2 = fit(p, w=1 / np.clip(X @ c, 1e-3, None))
    s2_ = 1.4826 * np.median(np.abs(r2 - np.median(r2)))
    keep = np.abs(r2) <= 3 * s2_
    q = p[keep].reset_index(drop=True)
    Xq = np.c_[np.ones(len(q)), (q['days'] / 365.25).to_numpy()]
    _, rq = fit(q)
    cq = np.linalg.lstsq(Xq, rq**2, rcond=None)[0]
    b3, _ = fit(q, w=1 / np.clip(Xq @ cq, 1e-3, None))
    touch_ = np.bincount(np.r_[q['q0'], q['q1']], minlength=T)
    out = 100 * np.exp(pd.Series(b3, index=labels))
    out[touch_ < min_touch] = np.nan                   # too few pairs: don't report
    out = out / out['2004Q1'] * 100
    return out, cq, len(p) - len(q), len(q)

city, cq, ntrim, nkept = pipeline(pairs)
print("citywide variance slope per year (robust basis):", cq[1].round(4), "| trimmed:", ntrim, "| kept:", nkept)

BORO = {1: 'Manhattan', 2: 'Bronx', 3: 'Brooklyn', 4: 'Queens', 5: 'Staten Is.'}
res = {'City': city}
for seg in ['HOUSE', 'CONDO_COOP', 'MULTIFAMILY']:
    res[seg], *_ = pipeline(pairs[pairs['SEGMENT'] == seg])
for b, nm in BORO.items():
    res[nm], *_ = pipeline(pairs[pairs['BOROUGH'] == b])
hpi = pd.DataFrame(res)

yrs_show = ['2004Q4','2007Q4','2010Q4','2013Q4','2016Q4','2019Q4','2020Q4','2022Q4','2025Q4']
print(hpi.loc[yrs_show].round(1).T.to_string())
hpi.to_pickle("../data/processed/hpi_by_group_quarterly.pkl")

citywide variance slope per year (robust basis): -0.0029 | trimmed: 20439 | kept: 343895
             2004Q4  2007Q4  2010Q4  2013Q4  2016Q4  2019Q4  2020Q4  2022Q4  2025Q4
City          114.8   149.3   118.8   145.6   175.8   195.7   198.8   213.9   228.0
HOUSE         113.5   143.5   103.6   115.1   147.8   180.0   188.7   215.8   250.1
CONDO_COOP    117.6   159.1   138.6   165.6   200.3   211.4   211.9   220.3   223.1
MULTIFAMILY   112.1   161.2   127.1   182.6   254.5   281.5   258.1   311.2   302.3
Manhattan     116.3   160.7   143.1   171.6   201.2   193.5   185.8   198.1   195.5
Bronx         117.8   158.9   110.5   119.4   144.7   189.8   201.9   217.8   249.2
Brooklyn      114.0   148.4   117.4   149.5   198.6   224.9   230.1   251.4   278.2
Queens        109.9   142.2   104.8   128.2   160.5   186.6   195.6   209.5   226.5
Staten Is.    113.6   127.7   109.9   112.0   136.0   160.0   164.7   189.0   216.2


In [6]:
EDGES = np.array([0, 0.5, 1, 2, 5, 10, 25.01])
mad_sd = lambda x: 1.4826 * np.median(np.abs(x - np.median(x)))

def pipeline2(p, min_touch=30):
    p = p.reset_index(drop=True)
    code = np.digitize(p['days'] / 365.25, EDGES) - 1
    _, r = fit(p)
    for _ in range(2):                                   # estimate per-gap-bucket noise, reweight
        sd_bin = np.array([mad_sd(r[code == k]) if (code == k).sum() >= 200 else mad_sd(r)
                           for k in range(len(EDGES) - 1)])
        _, r = fit(p, 1 / sd_bin[code] ** 2)
    keep = np.abs(r) / sd_bin[code] <= 3                 # trim on standardized residual
    q, cq = p[keep].reset_index(drop=True), code[keep]
    b, _ = fit(q, 1 / sd_bin[cq] ** 2)
    touch_ = np.bincount(np.r_[q['q0'], q['q1']], minlength=T)
    out = 100 * np.exp(pd.Series(b, index=labels))
    out[touch_ < min_touch] = np.nan
    return out / out['2004Q1'] * 100, sd_bin.round(3), int((~keep).sum()), len(q)

city, sd_bin, ntrim, nkept = pipeline2(pairs)
print("noise SD by gap bucket:", dict(zip(['<.5y', '.5-1', '1-2', '2-5', '5-10', '10+'], sd_bin)))
print("trimmed:", ntrim, "| kept:", nkept)

res = {'City': city}
for seg in ['HOUSE', 'CONDO_COOP', 'MULTIFAMILY']:
    res[seg] = pipeline2(pairs[pairs['SEGMENT'] == seg])[0]
for b, nm in BORO.items():
    res[nm] = pipeline2(pairs[pairs['BOROUGH'] == b])[0]
hpi = pd.DataFrame(res)
print(hpi.loc[yrs_show].round(1).T.to_string())
hpi.to_pickle("../data/processed/hpi_by_group_quarterly.pkl")

# Sensitivity: minimum holding period before a pair counts
sens = {f">={c}d": pipeline2(pairs[pairs['days'] >= c])[0] for c in [90, 180, 365]}
print(pd.DataFrame(sens).loc[yrs_show].round(1).T.to_string())

noise SD by gap bucket: {'<.5y': np.float64(0.335), '.5-1': np.float64(0.4), '1-2': np.float64(0.314), '2-5': np.float64(0.226), '5-10': np.float64(0.261), '10+': np.float64(0.301)}
trimmed: 21177 | kept: 343157
             2004Q4  2007Q4  2010Q4  2013Q4  2016Q4  2019Q4  2020Q4  2022Q4  2025Q4
City          114.8   149.4   120.2   147.2   179.0   199.0   202.7   216.9   232.1
HOUSE         112.8   142.8   102.8   115.8   147.3   179.8   187.9   214.6   251.6
CONDO_COOP    117.9   160.1   139.4   166.8   202.2   213.2   215.1   223.6   226.4
MULTIFAMILY   113.5   162.5   126.4   183.5   263.3   288.4   267.2   319.9   326.7
Manhattan     117.0   160.9   143.4   172.2   202.2   194.7   187.4   200.4   197.1
Bronx         115.7   157.0   109.0   119.3   143.7   189.0   205.8   215.8   247.9
Brooklyn      113.5   147.3   118.4   151.7   201.4   227.2   231.9   253.3   281.4
Queens        111.0   142.7   105.9   129.7   163.8   191.5   199.6   213.4   233.8
Staten Is.    113.1   126.0   11

In [9]:
cs = pd.read_csv("../data/raw/NYXRSA.csv")
cs.columns = ['date', 'cs']
cs['date'] = pd.to_datetime(cs['date']); cs['cs'] = pd.to_numeric(cs['cs'], errors='coerce')
cs = cs.dropna()
cs['q'] = cs['date'].dt.year.astype(str) + 'Q' + cs['date'].dt.quarter.astype(str)
csq = cs.groupby('q')['cs'].mean()
csq = csq / csq['2004Q1'] * 100

cmp_ = pd.DataFrame({'Ours_City': hpi['City'], 'Ours_HOUSE': hpi['HOUSE'], 'Case_Shiller_NY': csq}).dropna()
print("overlap:", cmp_.index[0], "to", cmp_.index[-1], "|", len(cmp_), "quarters")
print(cmp_.loc[[q for q in cmp_.index if q.endswith('Q4')]].round(1).to_string())

lv = np.log(cmp_)
print("\ncorrelation of QUARTERLY changes with Case-Shiller:")
print(lv.diff().dropna().corr()['Case_Shiller_NY'].round(3).to_string())
print("\ncorrelation of YEAR-OVER-YEAR changes with Case-Shiller:")
print(lv.diff(4).dropna().corr()['Case_Shiller_NY'].round(3).to_string())

overlap: 2003Q1 to 2025Q4 | 92 quarters
        Ours_City  Ours_HOUSE  Case_Shiller_NY
2003Q4       95.3        95.4             96.6
2004Q4      114.8       112.8            110.5
2005Q4      140.6       137.5            126.5
2006Q4      150.9       152.9            128.8
2007Q4      149.4       142.8            122.6
2008Q4      139.4       122.4            112.1
2009Q4      128.3       108.8            103.9
2010Q4      120.2       102.8            102.0
2011Q4      127.1       100.6             99.2
2012Q4      135.8       106.8             98.3
2013Q4      147.2       115.8            104.1
2014Q4      158.6       123.8            105.7
2015Q4      169.8       134.8            108.8
2016Q4      179.0       147.3            111.7
2017Q4      188.3       162.7            117.9
2018Q4      196.0       172.9            121.3
2019Q4      199.0       179.8            122.4
2020Q4      202.7       187.9            133.2
2021Q4      213.1       203.7            152.1
2022Q4      216.9   

In [10]:
cmp_ = pd.DataFrame({'City': hpi['City'], 'HOUSE': hpi['HOUSE'], 'CS': csq}).dropna()
lv = np.log(cmp_)

# 1. Does light smoothing of our series (CS is a 3-month moving average) raise the match?
for k in [1, 2, 4]:
    sm = np.log(cmp_[['City', 'HOUSE']].rolling(k).mean())
    both = pd.concat([sm.diff(4), lv['CS'].diff(4)], axis=1).dropna()
    print(f"rolling {k}q: YoY corr  City {both['City'].corr(both['CS']):.3f} | HOUSE {both['HOUSE'].corr(both['CS']):.3f}")

# 2. Where do we agree? Sub-periods
d4 = lv.diff(4).dropna()
for a, b in [('2004Q1', '2012Q4'), ('2013Q1', '2019Q4'), ('2020Q1', '2025Q4')]:
    x = d4.loc[a:b]
    print(a, b, "HOUSE corr", round(x['HOUSE'].corr(x['CS']), 3), "| City corr", round(x['City'].corr(x['CS']), 3), "| n", len(x))

# 3. Cumulative growth by era
for a, b in [('2004Q1', '2007Q4'), ('2007Q4', '2011Q4'), ('2011Q4', '2019Q4'), ('2019Q4', '2025Q4')]:
    g = [100 * (cmp_.loc[b, c] / cmp_.loc[a, c] - 1) for c in ['HOUSE', 'City', 'CS']]
    print(f"{a} -> {b}:  HOUSE {g[0]:+.0f}% | City {g[1]:+.0f}% | Case-Shiller {g[2]:+.0f}%")

# 4. House indexes by borough vs Case-Shiller (the metro is mostly suburban houses,
#    so the outer-borough house markets should be the closest comparison)
hb = pd.DataFrame({nm: pipeline2(pairs[(pairs['SEGMENT'] == 'HOUSE') & (pairs['BOROUGH'] == b)])[0]
                   for b, nm in BORO.items()}).loc[cmp_.index]
x = np.log(hb).diff(4)
x['CS'] = lv['CS'].diff(4)
print(x.corr()['CS'].round(3).to_string())

rolling 1q: YoY corr  City 0.721 | HOUSE 0.832
rolling 2q: YoY corr  City 0.707 | HOUSE 0.819
rolling 4q: YoY corr  City 0.631 | HOUSE 0.760
2004Q1 2012Q4 HOUSE corr 0.964 | City corr 0.959 | n 36
2013Q1 2019Q4 HOUSE corr 0.649 | City corr 0.428 | n 28
2020Q1 2025Q4 HOUSE corr 0.423 | City corr 0.685 | n 24
2004Q1 -> 2007Q4:  HOUSE +43% | City +49% | Case-Shiller +23%
2007Q4 -> 2011Q4:  HOUSE -30% | City -15% | Case-Shiller -19%
2011Q4 -> 2019Q4:  HOUSE +79% | City +57% | Case-Shiller +23%
2019Q4 -> 2025Q4:  HOUSE +40% | City +17% | Case-Shiller +66%
Manhattan     0.326
Bronx         0.791
Brooklyn      0.735
Queens        0.755
Staten Is.    0.862
CS            1.000


In [12]:
CNT = {'Bronx': 'ATNHPIUS36005A', 'Brooklyn': 'ATNHPIUS36047A', 'Queens': 'ATNHPIUS36081A',
       'Manhattan': 'ATNHPIUS36061A', 'Staten Is.': 'ATNHPIUS36085A'}
fh = {}
for nm, sid in CNT.items():
    d = pd.read_csv(f"../data/raw/{sid}.csv"); d.columns = ['date', 'v']
    d['year'] = pd.to_datetime(d['date']).dt.year
    d['v'] = pd.to_numeric(d['v'], errors='coerce')
    fh[nm] = d.set_index('year')['v']
fh = pd.DataFrame(fh)

ours = hpi[list(CNT)].copy()
ours['year'] = [int(l[:4]) for l in ours.index]
ours = ours.groupby('year').mean()

yrs = [y for y in ours.index if y in fh.index and y >= 2004]
print("years compared:", yrs[0], "to", yrs[-1])
print("NaN in ours:\n", ours.loc[yrs].isna().sum().to_string())
print("NaN in FHFA:\n", fh.loc[yrs].isna().sum().to_string())

r0 = lambda x: np.round(x, 0)
g = lambda s, a, b: 100 * (np.exp(s.get(b, np.nan) - s.get(a, np.nan)) - 1)
last = yrs[-1]
rows = []
for nm in CNT:
    o, f = np.log(ours.loc[yrs, nm]), np.log(fh.loc[yrs, nm])
    rows.append({'borough': nm,
                 'YoY_corr': round(o.diff().corr(f.diff()), 3),
                 '04-07 ours': r0(g(o, 2004, 2007)), '04-07 FHFA': r0(g(f, 2004, 2007)),
                 '11-19 ours': r0(g(o, 2011, 2019)), '11-19 FHFA': r0(g(f, 2011, 2019)),
                 '19-last ours': r0(g(o, 2019, last)), '19-last FHFA': r0(g(f, 2019, last))})
print(pd.DataFrame(rows).to_string(index=False))

years compared: 2004 to 2025
NaN in ours:
 Bronx         0
Brooklyn      0
Queens        0
Manhattan     0
Staten Is.    0
NaN in FHFA:
 Bronx         0
Brooklyn      0
Queens        0
Manhattan     7
Staten Is.    0
   borough  YoY_corr  04-07 ours  04-07 FHFA  11-19 ours  11-19 FHFA  19-last ours  19-last FHFA
     Bronx     0.905        45.0        42.0        57.0        34.0          30.0          38.0
  Brooklyn     0.866        38.0        42.0        84.0        54.0          22.0          22.0
    Queens     0.929        38.0        40.0        67.0        50.0          21.0          29.0
 Manhattan     0.369        41.0        78.0        35.0         NaN           0.0           NaN
Staten Is.     0.948        20.0        25.0        44.0        38.0          33.0          30.0


In [13]:
import numpy as np, pandas as pd
hpi = pd.read_pickle("../data/processed/hpi_by_group_quarterly.pkl")

# Case-Shiller NY vs our citywide and house indexes
cs = pd.read_csv("../data/raw/NYXRSA.csv"); cs.columns = ['date', 'cs']
cs['date'] = pd.to_datetime(cs['date']); cs['cs'] = pd.to_numeric(cs['cs'], errors='coerce'); cs = cs.dropna()
cs['q'] = cs['date'].dt.year.astype(str) + 'Q' + cs['date'].dt.quarter.astype(str)
csq = cs.groupby('q')['cs'].mean(); csq = csq / csq['2004Q1'] * 100
v = pd.DataFrame({'Ours City': hpi['City'], 'Ours HOUSE': hpi['HOUSE'], 'Case-Shiller NY': csq}).dropna()
v.index.name = 'quarter'; v = v.reset_index()
v['quarter_start'] = [f"{q[:4]}-{3 * (int(q[5]) - 1) + 1:02d}-01" for q in v['quarter']]
v.to_csv("../app/data/validation_case_shiller.csv", index=False)

# FHFA county indexes vs our borough indexes (annual, each rebased to its first year = 100)
CNT = {'Bronx': 'ATNHPIUS36005A', 'Brooklyn': 'ATNHPIUS36047A', 'Queens': 'ATNHPIUS36081A', 'Staten Is.': 'ATNHPIUS36085A'}
fh = {}
for nm, sid in CNT.items():
    d_ = pd.read_csv(f"../data/raw/{sid}.csv"); d_.columns = ['date', 'v']
    d_['year'] = pd.to_datetime(d_['date']).dt.year; d_['v'] = pd.to_numeric(d_['v'], errors='coerce')
    fh[nm] = d_.set_index('year')['v']
fh = pd.DataFrame(fh)
ours = hpi[list(CNT)].copy(); ours['year'] = [int(l[:4]) for l in ours.index]; ours = ours.groupby('year').mean()
rows = []
for nm in CNT:
    yrs = [y for y in ours.index if y in fh.index and y >= 2004 and pd.notna(fh.loc[y, nm])]
    o, f = ours.loc[yrs, nm], fh.loc[yrs, nm]
    rows.append(pd.DataFrame({'borough': nm, 'year': yrs, 'ours': (o / o.iloc[0] * 100).values, 'fhfa': (f / f.iloc[0] * 100).values}))
pd.concat(rows).to_csv("../app/data/validation_fhfa.csv", index=False)
print(len(v), sum(len(r) for r in rows))

92 88
